In [ ]:
import os
from eduray import *
from math import radians, tan
from dataclasses import field
from eduray.internals import *
from eduray.material import to_u8
from dataclasses import dataclass
from typing import List, Tuple
import random

# -------- setup for notebooks --------
if not os.path.exists("images"):
    try:
        os.makedirs("images")
    except FileExistsError:
        pass

random.seed(42)  # for reproducibility of random rays in examples

---
## Cameras and Rays

Now that we have a way to store an image, we can look at how a ray tracer views a scene. Every rendered image starts with a **camera**. Its job is to take a 2D pixel position and turn it into a 3D ray that travels into the scene.

## Pinhole camera model

We model the camera as an ideal **pinhole camera**. Each primary ray begins at the camera origin $\mathbf{o}$ and travels through a point on a flat **projection plane** placed at a
distance $d = 1$ in front of the camera.

### Field of view and projection plane size

The size of the projection plane depends on the field of view and the selected image aspect ratio.

Let $f$ be the vertical field-of-view angle in radians. Then the half-height of the projection plane is

$$
h = d \cdot \tan\!\left(\frac{f}{2}\right)
$$

This comes from the right triangle formed by the camera origin, the center of the projection plane, and its top edge.

The half-width is obtained from the image aspect ratio:

$$
r = \frac{w_{px}}{h_{px}},
\qquad
w = h \cdot r
$$

where $w_{px}$ and $h_{px}$ are the image width and height in pixels. This means that the
projection plane covers the interval $[-w, w]$ horizontally and $[-h, h]$ vertically but camera itself does not know anything about pixels. The pixel coordinates are only used to determine the ray direction, but the camera is defined in continuous space.

To generate a primary ray for a specific pixel, we map pixel coordinates to a point on the projection plane. A convenient approach is to first express the image-plane coordinates in
normalized form over the interval $[-1, 1]$, and then scale them by the corresponding half-width and half-height. The ray direction is then the vector from the camera origin to
that point on the projection plane.

This is different from rasterization, which works in the opposite direction. Instead of shooting rays from the camera into the scene, rasterization projects scene geometry, usually
triangles, onto the image plane and determines which pixels are covered. Ray tracing is therefore naturally expressed as a visibility problem from the camera, while rasterization is
based on projecting objects into screen space.

For an educational ray tracer, the pinhole camera is a good choice because it is simple and clearly shows how pixel coordinates map to rays, while more advanced camera models are left beyond the scope of this introductory notebook.

### Camera basis vectors

To place the camera correctly in world space, we build an **orthonormal basis**
$\{\mathbf{forward}, \mathbf{right}, \mathbf{up}\}$ from the viewing direction
$\mathbf{d}$ and an auxiliary up vector $\mathbf{up}_{hint}$:

$$
\mathbf{forward} = \frac{\mathbf{d}}{\|\mathbf{d}\|},
\qquad
\mathbf{right} = \frac{\mathbf{forward} \times \mathbf{up}_{hint}}
{\|\mathbf{forward} \times \mathbf{up}_{hint}\|},
\qquad
\mathbf{up} = \mathbf{right} \times \mathbf{forward}
$$

This basis defines the camera orientation and makes it possible to express points on the
projection plane in world coordinates.

With this setup, we can implement the `Camera` dataclass below. The `Visualizer` can then be
used to check whether the resulting frustum matches the chosen field of view.

## Rays representation

A ray is defined by an origin point $\mathbf{o}$ and a direction vector $\mathbf{d}$. The ray can be expressed parametrically as:
$$\mathbf{r}(t) = \mathbf{o} + t\,\mathbf{d}
$$
where $t \geq 0$ is a parameter that controls how far along the ray we are. The ray starts at $\mathbf{o}$ when $t=0$ and extends infinitely in the direction of $\mathbf{d}$ as $t$ increases.

In this ray tracer, the `Ray` class encapsulates this concept, storing the origin and direction as attributes. The `make_ray` method of the `Camera` class generates rays based on uv



In [ ]:
@dataclass
class MyPinholeCamera(Camera):
    """
    Simple pinhole camera model with position, direction, up vector and field of view. The camera generates rays for each pixel on the image plane based on its orientation and FOV.
    """

    # Camera parameters
    fov_deg: float = 70.0
    origin: Vertex = field(default_factory=lambda: Vertex(0, 0, 0))
    direction: Vector = field(default_factory=lambda: Vector(0, 0, -1))

    # orthonormal basis
    up_hint: Vector = field(default_factory=lambda: Vector(0, 1, 0))
    forward: Vector = field(init=False)
    right: Vector = field(init=False)
    up: Vector = field(init=False)

    aspect_ratio: float = 16.0 / 9.0

    half_width: float = field(init=False)
    half_height: float = field(init=False)

    def __post_init__(self):
        self.update_camera()

    def update_camera(self):
        """
        Calculate camera basis vectors and image plane dimensions.
        :return: None
        """
        forward = self.direction.normalize()
        up = self.up_hint
        # if direction is parallel to up_hint, choose a different up vector
        if abs(forward.dot(up)) > 0.999:
            up = Vector(1, 0, 0)

        # build orthonormal basis right-handed system - camera looks along -Z
        w = -forward
        right = up.cross(w).normalize()
        true_up = w.cross(right)

        # calculate image plane dimensions based on FOV and aspect ratio
        theta = radians(self.fov_deg)
        half_height = tan(theta * 0.5)
        half_width = self.aspect_ratio * half_height

        self.forward = forward
        self.right = right
        self.up = true_up
        self.half_width = half_width
        self.half_height = half_height

    def make_ray(self, u: float, v: float) -> Ray:
        """
        u, v in [-1, 1]
        (-1,-1)=bottom-left, (1,1)=top-right
        Image plane is 1 unit in front of the camera.
        """
        center_plane = self.origin + self.forward

        position = (
                center_plane
                + self.right * (u * self.half_width)
                + self.up * (v * self.half_height)
        )
        return Ray(self.origin, (position - self.origin).normalize())

    def set_aspect_ratio(self, aspect_ratio: float):
        self.aspect_ratio = aspect_ratio
        self.update_camera()

    def copy(self) -> Camera:
        return MyPinholeCamera(
            fov_deg=self.fov_deg,
            aspect_ratio=self.aspect_ratio,
            origin=self.origin,
            direction=self.direction,
            up_hint=self.up_hint
        )

    def zoom(self, factor: float) -> None:
        self.fov_deg *= factor
        self.update_camera()

---
## Camera orientation and frustum

Before generating images, it is helpful to view the camera as a geometric object embedded in
the scene. The visualization below highlights three key components:

- **Origin** — the single point from which all primary rays are emitted
- **Basis vectors** — `forward`, `right`, and `up` define the orthonormal frame that orients
  the camera in world space. Here, `right = up_hint × forward` and `up = forward × right`,
  ensuring that all three axes remain mutually perpendicular
- **Frustum** — the pyramid-shaped viewing volume of the camera, whose width is determined by
  the field of view

> **Try it:** change `fov_deg` and observe how the frustum changes shape. A larger field of view
produces a wider and more distorted view, while a smaller field of view narrows the visible
region similarly to a telephoto lens.

In [ ]:
camera = MyPinholeCamera(
    origin=Vertex(4, 2, 0),
    direction=Vector(-1, -0.3, 0),
    fov_deg=80,
    aspect_ratio=Resolution.R360p.aspect_ratio
)

vis = Visualizer()
vis.create_empty_scene(size=4.0,figsize=(12, 8), show_axes_labels=False, show_arrows=True, show_grid=True, show_axes=True, show_xyz_labels=True)
vis.visualize_camera_position_and_orientation(camera, show_frustum=False, frustum_depth=5.0, show_camera_orientation=True)
vis.savefig("images/camera_position_and_orientation.png", dpi=300, fontsize=15)
vis.show()

### Frustum visualization

In [ ]:
vis.create_empty_scene(size=3.5, figsize=(12, 8), show_axes_labels=False, show_arrows=True, show_grid=True, show_axes=True, show_xyz_labels=True)
vis.visualize_camera_position_and_orientation(camera, show_frustum=True, frustum_depth=5.0, show_camera_orientation=True)
vis.savefig("images/fov_80_camera_frustum.pdf", dpi=300, fontsize=13)
vis.show()

### Different field of view

In [ ]:
camera.fov_deg = 20
camera.update_camera()

vis.create_empty_scene(size=3.5, figsize=(12, 8), show_axes_labels=False, show_arrows=True, show_grid=True, show_axes=True, show_xyz_labels=True)
vis.visualize_camera_position_and_orientation(camera, show_frustum=True, frustum_depth=5.0, show_camera_orientation=True)
vis.savefig("images/fov_20_camera_frustum.pdf", dpi=300, fontsize=13)
vis.show()

# now we return to 70 degrees for the rest of the notebook
camera.fov_deg = 70
camera.update_camera()

### UV coordinates

> **Try it:** change `uv` and observe how the generated ray changes direction. The `uv` coordinates determine where on the projection plane the ray passes through. For example, `uv=(-1, -1)` corresponds to the bottom-left corner of the projection plane, while `uv=(1, 1)` corresponds to the top-right corner. Changing these values will cause the ray to point in different directions, allowing you to see how rays are generated across the field of view of the camera.

In [ ]:
u = -0.3
v = 0.5

vis.create_empty_scene(size=2.0, figsize=(12, 8), show_axes_labels=False, show_arrows=True, show_grid=True, show_axes=False, show_xyz_labels=True, view_elev=20, view_azim=0)
vis.visualize_camera_position_and_orientation(camera, show_frustum=False, frustum_depth=4.0, show_camera_orientation=False, show_plane=True, show_plane_corners=True)
vis.visualize_image_plane_point(camera, u=u, v=v, label=True)
vis.savefig("images/projection_plane.pdf", dpi=300, fontsize=15)
vis.show()

### Aspect ratio

> **Try it:** Change `aspect_ratio` and observe how the frustum changes. In this implementation, the field of view is defined vertically, so changing the aspect ratio mainly affects the **horizontal** field of view. A wider aspect ratio, such as `16:9`, produces a wider horizontal view, while a narrower one, such as `4:3`, reduces the horizontal view and makes the scene appear more zoomed in from left to right.
>
> Because the projection plane is scaled differently in the horizontal and vertical directions, the viewing volume does not have the same extent along both axes. As a result, the image may appear wider in one direction and narrower in the other.
>
> In this library, the aspect ratio is set internally by the render loop according to the selected output resolution. This means that changing the resolution also changes the aspect ratio used during rendering.

In [ ]:
camera.aspect_ratio = Resolution.R256p_1_1.aspect_ratio
camera.update_camera()

vis.create_empty_scene(size=3.5, figsize=(12, 8), show_axes_labels=False, show_arrows=True, show_grid=True, show_axes=True, show_xyz_labels=True)
vis.visualize_camera_position_and_orientation(camera, show_frustum=True, frustum_depth=5.0, show_camera_orientation=True)
vis.savefig("images/aspect_ratio_1_1.pdf", dpi=300, fontsize=13)
vis.show()

In [ ]:
camera.aspect_ratio = Resolution.R360p.aspect_ratio
camera.update_camera()

vis.create_empty_scene(size=3.5, figsize=(12, 8), show_axes_labels=False, show_arrows=True, show_grid=True, show_axes=True, show_xyz_labels=True)
vis.visualize_camera_position_and_orientation(camera, show_frustum=True, frustum_depth=5.0, show_camera_orientation=True)
vis.savefig("images/aspect_ratio_16_9.pdf", dpi=300, fontsize=13)
vis.show()

## From pixel $(i, j)$ to a ray direction $(u, v)$

Each pixel $(i, j)$ is mapped to a point on the projection plane using **normalized device
coordinates** (NDC):

$$
u = \frac{i + 0.5}{w_{px}} \cdot 2 - 1,
\qquad
v = 1 - \frac{j + 0.5}{h_{px}} \cdot 2
$$

The offset $+0.5$ places the sample at the **center** of the pixel. The formula for $v$ also flips the $Y$ axis, converting image coordinates, which grow from top to bottom, into
camera coordinates, where the vertical axis points upward.

We then use $(u, v)$ to select a point on the projection plane. The ray direction is given by the vector from the camera origin $\mathbf{o}$ to that point $\mathbf{p}$, normalized to
unit length:

$$
\mathbf{d}_{ray} = \frac{\mathbf{p} - \mathbf{o}}{\|\mathbf{p} - \mathbf{o}\|}
$$

This gives one primary ray for each pixel sample and forms the basic idea behind image generation in the pinhole camera model.

> **Try it:** Change the mapping from pixel coordinates to $(u, v)$. For example, what happens if you remove the `+0.5` offset? What if you do not flip the $v$ coordinate? Observe how
> these changes affect the resulting image.

In [ ]:
vis.create_empty_scene(size=4.0, figsize=(12, 8), show_axes_labels=False, show_arrows=False, show_grid=True, show_axes=False, show_xyz_labels=True)
vis.visualize_camera_position_and_orientation(camera, show_frustum=True, frustum_depth=6, show_camera_orientation=False, show_plane=True, show_plane_corners=False)

WIDTH = 10
HEIGHT = 10

for x in range(WIDTH):
    for y in range(HEIGHT):
        u = (x + 0.5) / WIDTH * 2 - 1   # centers
        v = 1 - (y + 0.5) / HEIGHT * 2
        vis.visualize_image_plane_point(camera, u, v, size=3)
        ray = camera.make_ray(u, v)
        vis.visualize_ray(ray, opacity=0.3)

vis.show_legend()
vis.savefig("images/rays.pdf", dpi=300, fontsize=13)
vis.show()

## The Render Loop

In a real render loop, rays are usually **not** fired randomly. In this notebook, however, some examples use random rays to make the basic idea of ray tracing easier to see and understand.
In practice, the standard approach is to generate rays in a **deterministic order** that matches the pixel layout of the output image. This process is commonly called the **render loop**.

In its simplest form, each pixel receives exactly one primary ray, or more if super sampling is enabled. The loop goes through the image in raster order, generates a ray through the
center of each pixel, traces that ray into the scene, computes the resulting color from object intersections and light contributions, and stores the final value in the output image.

The pseudocode for a simple render loop is shown below:

```
for j in range(height):          # rows top → bottom
    for i in range(width):        # columns left → right
        ray    = camera.make_ray(u, v)
        color = algorithm_for_color_calculation(ray)
        image[j, i] = color
```

### Simple loop

We begin with a basic render loop that processes the image sequentially, pixel by pixel, from left to right and from top to bottom. This is the simplest rendering strategy and
serves as a useful starting point for understanding the overall rendering pipeline.

For each pixel `(i, j)`, the raster coordinates are first converted to normalized image-plane coordinates `u` and `v`. The camera then generates a primary ray from these
coordinates, and this ray is passed to the integrator, which computes the resulting color.
The floating-point color values are then converted to 8-bit RGB values so they can be stored in the output image as PPM format then converted to PNG for display.

The `render_all_pixels` method iterates over the entire image in row-major order, calls `render_pixel` for each pixel, and stores the resulting RGB triplets in a flat list.
This list can later be written directly to an image file format such as PPM.

This approach is intentionally simple and easy to follow. Its goal is not performance, but clarity.
It provides a clear reference implementation of the render loop before moving on to
more advanced strategies such as random pixel order, progressive refinement, or other sampling techniques.

In [ ]:
@dataclass
class MySimpleRenderLoop(RenderLoop):
    """
    Simple render loop
    """
    def render_pixel(self, i: int, j: int) -> Tuple[int, int, int]:
        u = (i + 0.5) / self.width * 2 - 1
        v = 1 - (j + 0.5) / self.height * 2

        ray = self.camera.make_ray(u , v )

        color = self.integrator.cast_ray(ray=ray, depth=self.max_depth)

        return to_u8(color.r), to_u8(color.g), to_u8(color.b) # saving values as RGB because of ppm format

    def render_all_pixels(self) -> Tuple[List[Tuple[int, int, int]], int, int]:

        pixels: List[Tuple[int, int, int]] = []
        total = self.width * self.height

        self.ui.start(total)

        for row in range(self.height):
            for i in range(self.width):
                rgb = self.render_pixel(i, row)
                pixels.append(rgb)

        return pixels, self.width, self.height

We now test the simple render loop on a minimal scene with a single sphere. In this example, the `NormalShader` is used instead of a standard lighting model, so the image shows
surface normals encoded as RGB colors. This makes it easier to check whether intersections and normal computation work correctly.

Preview is disabled here, because this basic loop does not yet support incremental updates. The next step will extend the render loop with preview updates and supersampling.

In [ ]:
sphere = Object(
    geometry=Sphere(center=Vertex(0, 0.5, -0.5), radius=1.0),
    material=PhongMaterial(base_color=Color.custom_rgb(0, 0, 0)),
)

rt = MySimpleRenderLoop(
    scene=Scene(
        camera=camera,
        objects=[sphere]
    ),
    render_config=RenderConfig(resolution=Resolution.R360p, max_depth=1),
    shading_model=NormalShader()
)
rendered_image = rt.render("images/RGB_normal_map.png")
ipynb_display_images("images/RGB_normal_map.png")

### Supersampling loop

This version extends the basic render loop by tracing multiple rays for each pixel.
Instead of using only one ray through the pixel center, it generates several rays with small
random offsets inside the pixel and averages their returned colors.

This reduces aliasing and produces a smoother image. The loop also keeps support for progress
display and preview updates after completed rows.

In [ ]:
@dataclass
class MySuperSamplingRenderLoop(RenderLoop):
    """
    Render loop with super sampling and support for progress display
    """
    def render_pixel(self, i: int, j: int) -> Tuple[int, int, int]:
        u = (i + 0.5) / self.width * 2 - 1
        v = 1 - (j + 0.5) / self.height * 2
        acc = Color.custom_rgb(0, 0, 0)

        for _ in range(self.spp):
            du = (random.random() - 0.5) * 2 / self.width
            dv = (random.random() - 0.5) * 2 / self.height

            ray = self.camera.make_ray(u + du, v + dv)

            acc += self.integrator.cast_ray(ray=ray, depth=self.max_depth)

        col = acc / self.spp
        return to_u8(col.r), to_u8(col.g), to_u8(col.b)

    def render_all_pixels(self) -> Tuple[List[Tuple[int, int, int]], int, int]:
        pixels: List[Tuple[int, int, int]] = []
        total = self.width * self.height

        self.ui.start(total)

        for row in range(self.height):
            for i in range(self.width):
                rgb = self.render_pixel(i, row)
                pixels.append(rgb)

        #helper functions for progress bar display and preview updates
            self.on_row_end_update_preview(row, pixels)
            self.ui.update_pixel(self.width)  # update progress bar after each row
        self.ui.update_end(pixels)

        return pixels, self.width, self.height

Unlike the basic loop, this version also supports preview updates during rendering.
Depending on the selected `ProgressDisplay` mode, progress output can be disabled, shown as a progress bar, or displayed as an image preview updated during rendering. In this example,
the `TQDM_IMAGE_PREVIEW` mode is used.

In [ ]:
rt = MySuperSamplingRenderLoop(
    scene=Scene(
        camera=camera,
        objects=[sphere]
    ),
    render_config=RenderConfig(resolution=Resolution.R360p, samples_per_pixel=4, max_depth=1),
    preview_config=PreviewConfig(progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW),
    shading_model=NormalShader()
)
image = rt.render("images/RGB_normal_map.png")
ipynb_display_images(image)

---
# Bonus

This variant renders pixels in a random order instead of the usual row-by-row traversal. As a result, the preview reveals the image gradually across the whole screen rather than
filling it one row at a time. This can make intermediate results easier to inspect and gives a more progressive appearance.

> **Try it:** change process of pixel generation to see how it affects the preview. For example, you could try rendering pixels in a spiral pattern, or by randomly shuffling the order of rows and columns. Observe how these different traversal strategies impact the way the image is revealed during rendering.

In [ ]:
@dataclass
class RandomPixelRenderLoop(RenderLoop):
    def render_pixel(self, i: int, j: int) -> Tuple[int, int, int]:
        u = (i + 0.5) / self.width * 2 - 1
        v = 1 - (j + 0.5) / self.height * 2

        ray = self.camera.make_ray(u, v)
        color = self.integrator.cast_ray(ray=ray, depth=self.max_depth)

        return to_u8(color.r), to_u8(color.g), to_u8(color.b)

    def render_all_pixels(self) -> Tuple[List[Tuple[int, int, int]], int, int]:
        pixels: List[Tuple[int, int, int]] = [(0, 0, 0)] * (self.width * self.height)
        coords = [(i, j) for j in range(self.height) for i in range(self.width)]
        random.shuffle(coords)

        total = self.width * self.height
        self.ui.start(total)

        interval_pixels = self.ui.preview.refresh_interval_rows * self.width

        for num_rendered, (i, j) in enumerate(coords, start=1):
            rgb = self.render_pixel(i, j)
            index = j * self.width + i
            pixels[index] = rgb

            self.ui.update_pixel(1)

            if (
                self.ui.img_widget is not None
                and (
                    num_rendered % interval_pixels == 0
                    or num_rendered == total
                )
            ):
                self.ui.update_image(pixels, rendered_pixels=num_rendered)

        self.ui.update_end(pixels)
        return pixels, self.width, self.height

### Rendering with random pixel order and preview updates

In [ ]:
rt = RandomPixelRenderLoop(
    scene=Scene(
        camera=camera,
        objects=[sphere]
    ),
    render_config=RenderConfig(resolution=Resolution.R360p, samples_per_pixel=1, max_depth=1),
    preview_config=PreviewConfig(progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW),
    shading_model=NormalShader()
)
image = rt.render("images/RGB_normal_map.png")
ipynb_display_images(image)

# Summary

In this notebook, we introduced the basic building blocks of a ray tracer:

- **Pinhole camera** — position, orthonormal basis ($\mathbf{forward}$, $\mathbf{right}$, $\mathbf{up}$), FOV, projection plane
- **Ray generation** — pixel $(i, j)$ → NDC $(u, v)$ → normalised ray direction
- **Render loop** — sequential, super sampled, and randomised pixel traversal strategies that you can experiment with

**Next:**
- **Ray–sphere intersection** — quadratic formula, discriminant cases, surface normal
- **Multiple primitives** — sphere, box, triangle, plane, cylinder, torus
- **Implicit surfaces** — SDFs, sphere tracing, finite-difference normals

---
## References
- Shirley, P., Black, T. D. & Hollasch, S. (2025).
  *Ray Tracing in One Weekend*. https://raytracing.github.io/
- Hughes, J. F., van Dam, A., et al. (2014). *Computer Graphics:
  Principles and Practice* (3rd ed.). Addison-Wesley.
- Pharr, M., Jakob, W. & Humphreys, G. (2016). *Physically Based Rendering: From Theory to Implementation* (3rd ed.).
  See also https://www.pbr-book.org/ for the online 4th edition.
- [Scratchapixel](https://www.scratchapixel.com/)